In [3]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("data/train.csv")

label_cols = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

df["is_harmful"] = (df[label_cols].sum(axis=1) > 0).astype(int)
df["comment_length"] = df["comment_text"].astype(str).str.len()

print(df.shape)
df.head()

(159571, 10)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,is_harmful,comment_length
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,0,264
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,0,112
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,0,233
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,0,622
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,0,67


In [4]:
summary = {
    "total_comments": len(df),
    "harmful_comments": int(df["is_harmful"].sum()),
    "harmful_rate": round(df["is_harmful"].mean(), 4)
}

summary

{'total_comments': 159571,
 'harmful_comments': 16225,
 'harmful_rate': np.float64(0.1017)}

In [5]:
label_cols = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

label_summary = df[label_cols + ["is_harmful"]].mean().sort_values(ascending=False)
label_summary

is_harmful       0.101679
toxic            0.095844
obscene          0.052948
insult           0.049364
severe_toxic     0.009996
identity_hate    0.008805
threat           0.002996
dtype: float64

In [6]:
import os

os.makedirs("outputs", exist_ok=True)

pd.DataFrame([summary]).to_csv("outputs/eda_summary.csv", index=False)
label_summary.to_csv("outputs/label_distribution.csv")

df[["id", "comment_text", "is_harmful", "comment_length"] + label_cols].to_csv(
    "outputs/processed_comments.csv",
    index=False
)

print("Files saved successfully.")

Files saved successfully.


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 准备数据
X = df["comment_text"].astype(str)
y = df["is_harmful"]

# 切分数据
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 向量化
vectorizer = TfidfVectorizer(max_features=10000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 模型
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# 预测
y_pred = model.predict(X_test_vec)

# 输出报告
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.99      0.98     28671
           1       0.93      0.64      0.75      3244

    accuracy                           0.96     31915
   macro avg       0.94      0.82      0.87     31915
weighted avg       0.96      0.96      0.95     31915



In [8]:
from sklearn.metrics import precision_score, recall_score, f1_score

# predicted probability for harmful class
y_score = model.predict_proba(X_test_vec)[:, 1]

threshold_results = []

for threshold in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    y_pred_threshold = (y_score >= threshold).astype(int)
    
    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_test, y_pred_threshold),
        "recall": recall_score(y_test, y_pred_threshold),
        "f1": f1_score(y_test, y_pred_threshold),
        "flagged_comments": int(y_pred_threshold.sum())
    })

threshold_df = pd.DataFrame(threshold_results)
threshold_df

,threshold,precision,recall,f1,flagged_comments
0,0.2,0.729104,0.825524,0.774324,3673
1,0.3,0.828114,0.750000,0.787124,2938
2,0.4,0.887475,0.688039,0.775135,2515
3,0.5,0.927575,0.635635,0.754344,2223
4,0.6,0.951070,0.575216,0.716865,1962
5,0.7,0.969628,0.521578,0.678292,1745
6,0.8,0.982747,0.456535,0.623448,1507
7,0.9,0.993399,0.371147,0.540395,1212


In [9]:
results_df = pd.DataFrame({
    "comment_text": X_test.values,
    "actual_label": y_test.values,
    "risk_score": y_score
})

def assign_action(score):
    if score >= 0.90:
        return "Auto Remove"
    elif score >= 0.50:
        return "Human Review"
    elif score >= 0.20:
        return "Monitor"
    else:
        return "Allow"

results_df["recommended_action"] = results_df["risk_score"].apply(assign_action)
results_df["predicted_label_05"] = (results_df["risk_score"] >= 0.5).astype(int)

results_df.to_csv("outputs/model_predictions.csv", index=False)
threshold_df.to_csv("outputs/threshold_analysis.csv", index=False)

results_df.head()

,comment_text,actual_label,risk_score,recommended_action,predicted_label_05
0,"Geez, are you forgetful! We've already discus...",0,0.258594,Monitor,0
1,Carioca RFA \n\nThanks for your support on my ...,0,0.007131,Allow,0
2,"""\n\n Birthday \n\nNo worries, It's what I do ...",0,0.063687,Allow,0
3,Pseudoscience category? \n\nI'm assuming that ...,0,0.005312,Allow,0
4,"(and if such phrase exists, it would be provid...",0,0.010670,Allow,0


In [10]:
false_positives = results_df[
    (results_df["actual_label"] == 0) & 
    (results_df["predicted_label_05"] == 1)
].sort_values("risk_score", ascending=False)

false_negatives = results_df[
    (results_df["actual_label"] == 1) & 
    (results_df["predicted_label_05"] == 0)
].sort_values("risk_score", ascending=False)

print("False Positives:", len(false_positives))
print("False Negatives:", len(false_negatives))

false_positives.head(10)

False Positives: 161
False Negatives: 1182


,comment_text,actual_label,risk_score,recommended_action,predicted_label_05
16127,wtf \n\nwhat is your problem with me are you a...,0,0.964611,Auto Remove,1
3878,You callin' me a liar?!,0,0.962394,Auto Remove,1
2073,"""\n\nwtf? ~ () (TCE) """,0,0.956413,Auto Remove,1
25671,voidid or whatever his name is. \n\nhey voidid...,0,0.946446,Auto Remove,1
4836,What did I do that was vandalism first of all ...,0,0.939162,Auto Remove,1
21884,Hi \n\nLuke's Ass.\n\nHow you doin' ?,0,0.928840,Auto Remove,1
8228,what is up with you editing all this less than...,0,0.919391,Auto Remove,1
12230,"Hey, im ripped as hell man",0,0.905775,Auto Remove,1
10014,oh wow a block im so scared. <sarcasm. stop ge...,0,0.893411,Human Review,1
25362,"You must think I'm a pestering loser, right? -...",0,0.885232,Human Review,1


In [11]:
false_negatives.head(10)

,comment_text,actual_label,risk_score,recommended_action,predicted_label_05
2784,00frodo \nI cant believe you gave me my own pa...,1,0.498572,Monitor,0
31111,What a lying BITCH!!! Look at this shyt \n\nht...,1,0.498292,Monitor,0
22921,"Why am I even talking to this retard, who hasn...",1,0.497190,Monitor,0
8551,You deleted the message I put here for you. Ge...,1,0.496940,Monitor,0
25176,"""\n\n And why is """"screw you, motherfucker"""" a...",1,0.494434,Monitor,0
25218,YO\nWHY CANT I MESS UP UR SITE\nNO SCHOOLS R A...,1,0.493239,Monitor,0
8348,Hell yes. Absolutely. \n\n==Please remove the...,1,0.492513,Monitor,0
23640,It is you who are the TROLL FReepsbane. You co...,1,0.492484,Monitor,0
14655,"""\nAfter several edit conflicts - Liz, you are...",1,0.492436,Monitor,0
296,"""\nThe Graceful Slick....\nIs non other than a...",1,0.491291,Monitor,0


In [12]:
false_positives.to_csv("outputs/false_positives.csv", index=False)
false_negatives.to_csv("outputs/false_negatives.csv", index=False)

print("Error analysis files saved.")

Error analysis files saved.


In [13]:
false_positives.to_csv("outputs/false_positives.csv", index=False)
false_negatives.to_csv("outputs/false_negatives.csv", index=False)

print("Error analysis files saved.")

Error analysis files saved.
